# Explanation-disagreement protocol pilot — BDHS 2022 child stunting

**What this notebook does:** one pre-specified run of the full protocol on BDHS 2022 (Kids Recode, `BDKR81FL.DTA`):
LR / RF / MLP → SHAP, LIME, Integrated Gradients (where mathematically valid) → dual-seed noise floor →
cross-method disagreement relative to that floor → null calibration → negative-control column → gate report.

**Version v3** (after pilot v2 + independent audit): SHAP background fixed per fold (plus a separate background-sensitivity axis), per-child-minimum conservative floors and G7, explicit undefined-κ handling, float64 LR twin, single-threaded torch / serial RF prediction, pinned libraries with fold-hash check, G0/G1 made non-vacuous.

**What it is NOT:** a model-selection or feature-search exercise. Nothing here is tuned on results.
If the gate report passes, the same code is run unchanged with `MODE = "FULL"`.

**Before you run:** if Colab asks to restart the runtime after the install cell, restart and continue from the settings cell. Upload `BDKR81FL.DTA` to the Colab file panel (`/content`). Edit only the *PRE-REGISTERED SETTINGS* cell, and only before the first real run.

**Data-use note:** the notebook writes only variable-level attribution tables and summaries. It does not save case IDs, cluster IDs or raw rows. Do not upload the DHS file anywhere else.

| | SHAP (variable-level Permutation SHAP) | LIME (tabular) | Integrated Gradients |
|---|---|---|---|
| **Logistic regression** | yes | yes | yes — via a float64 PyTorch twin of the fitted sklearn model (same function; checked in G4c) |
| **Random forest** | yes | yes | **not applicable** (non-differentiable) |
| **MLP (PyTorch)** | yes | yes | yes (Captum) |

In [ ]:
# Colab only. The explainer libraries are PINNED to the pilot-v2 environment (shap 0.52.0, lime 0.2.0.1, captum 0.9.0). The pilot ran on Colab Python 3.13 / numpy 2.1.3 / pandas 2.2.3 / scikit-learn 1.6.1 / torch 2.11.0+cpu;
# run_metadata.json records what actually ran, and the fold hash check in section 3 fails if scikit-learn changed the cluster-grouped split.
!pip -q install "shap==0.52.0" "lime==0.2.0.1" "captum==0.9.0"

## 1. PRE-REGISTERED SETTINGS
Edit only before the first real run. Every value here is written to `run_metadata.json`.

In [ ]:
import os, sys, glob, json, time, math, warnings, itertools, platform, random
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

MODE = "PILOT"            # "SMOKE" (2-3 min wiring test) | "PILOT" | "FULL"
FEATURE_SET = "S0"        # "S0" = 14 background variables (primary). "S5" = S0 + v438 (maternal height; exploratory, needs advisor approval)

# --- seeds (fresh split; the earlier pilot used 20261007) ---
SEED_SPLIT, SEED_INST, SEED_NOISE, SEED_BOOT, SEED_NULL = 20261016, 20261017, 20261018, 20261019, 20261020
SEED_BG = 20261021                                # SHAP background subsample: fixed per fold (SEED_BG + fold), so the E-axis floor isolates permutation randomness
WORK = {}                                         # measured seconds of scheduled work (probe, main run, controls, TreeSHAP check) -> gate G1
R_E, R_T = 5, 5                                   # explainer-seed runs, training-seed runs
ES = [1001 + i for i in range(R_E)]               # explainer seeds
TS = [2001 + i for i in range(R_T)]               # training seeds

# --- explanation budgets (identical in PILOT and FULL; only the number of explained children differs) ---
N_PER_FOLD = {"SMOKE": 4, "PILOT": 20, "FULL": 40}[MODE]
N_PERM, B_SHAP = 10, 50                           # Permutation-SHAP permutations (forward+reverse pass each) and background rows
LIME_SAMPLES   = 5000
IG_STEPS       = 100
TOPK           = 5                                # top-k for Jaccard (of 14 variables)
if MODE == "SMOKE":
    R_E, R_T, N_PERM, LIME_SAMPLES, IG_STEPS = 3, 3, 3, 500, 30
    ES, TS = ES[:R_E], TS[:R_T]

# --- fixed model settings (taken from the earlier pilot's tuning; NOT re-tuned here) ---
HP = dict(LR=dict(C=0.1), RF=dict(n_estimators=300, max_depth=5, min_samples_leaf=5),
          MLP=dict(hidden=32, epochs=90, lr=1e-3, wd=1e-4, batch=128))
N_OUTER_FOLDS = 5

# --- gate thresholds (proposed; edit BEFORE running, never after seeing results) ---
GATES = dict(
    max_projected_hours=4.0,          # G1 compute: projected total run time
    min_auc_weighted=0.58,            # G2 models are better than trivial (survey-weighted AUROC, mean over folds)
    max_auc_gap_vs_earlier=0.04,      # G2b reproduces earlier S0 AUROCs (LR .624, RF .644, MLP .621) within this gap
    null_spearman_abs=0.05,           # G3 random-ranking null Spearman must be ~0
    max_additivity_err=1e-3,          # G4 SHAP local accuracy
    max_ig_delta=1e-2,                # G4 IG completeness
    noise_min_median_rank=11,         # G5 negative control: noise column median rank must be >= 11 of 15 (1 = most important) ...
    noise_max_share_ratio=0.5,        # ... and its mean importance share must be <= 0.5 x the uniform share (1/15)
    max_kappa_ci_width=0.30,          # G6 precision: 95% CI width of kappa (Spearman, SHAP vs LIME) per model
    min_floor_minus_null=0.05,        # G7 floor must exceed the null by this much, else kappa is unstable
    max_tie_share=0.01,               # G8 share of attribution vectors that are all-zero or tied at the top-k boundary
)
EARLIER_S0_AUC = dict(LR=0.624, RF=0.644, MLP=0.621)

# --- variables (order = earlier manifest) ---
S0_VARS = ["b4", "b19", "bord", "v012", "v106", "v190", "v025", "v024", "v130", "v136", "v151", "v113", "v116", "v212"]
CAT_VARS_ALL = ["b4", "v025", "v024", "v130", "v151", "v113", "v116"]
VARS = S0_VARS + (["v438"] if FEATURE_SET == "S5" else [])
CAT_VARS = [v for v in VARS if v in CAT_VARS_ALL]
SPECIAL_CAT_MISSING = {"v113": [97], "v116": [97]}       # 97 = not a de-jure resident -> structural-missing level (-1). v116 code 15 ("flush, don't know where") is a real answer and is KEPT as its own level (the earlier pilot merged it into missing; deviation recorded)
DATA_PATH = os.environ.get("THESIS_DATA_PATH")           # leave None: auto-search /content
OUT_DIR = "/content/outputs_protocol_pilot" if os.path.isdir("/content") else "./outputs_protocol_pilot"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODE={MODE}  FEATURE_SET={FEATURE_SET}  explained children/fold={N_PER_FOLD}  runs: R_E={R_E}, R_T={R_T}")

## 2. Libraries, reproducibility, data

In [ ]:
import torch, torch.nn as nn
import shap, lime, lime.lime_tabular
from captum.attr import IntegratedGradients
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from importlib import metadata as _md0
def _v0(p):
    try: return _md0.version(p)
    except Exception: return 'n/a'
PINNED = dict(shap='0.52.0', lime='0.2.0.1', captum='0.9.0')
_drift = {p: (v, _v0(p)) for p, v in PINNED.items() if _v0(p) != v}
if _drift and os.environ.get('THESIS_ALLOW_VERSION_DRIFT') != '1':
    raise RuntimeError(f'Explainer library versions differ from the pre-registered pins (pinned, found): {_drift}. Re-run the install cell and restart the runtime.')
print('explainer libraries:', {p: _v0(p) for p in PINNED}, '| sklearn', sklearn.__version__, '| numpy', np.__version__, '| torch', torch.__version__)

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_all_seeds(SEED_SPLIT)
torch.set_num_threads(1)      # single thread: float summation order (hence trained MLP weights, IG) must not depend on the host's core count

In [ ]:
def find_dta():
    if DATA_PATH and os.path.exists(DATA_PATH): return DATA_PATH
    pats = ["/content/*.[dD][tT][aA]", "/content/*/*.[dD][tT][aA]", "./*.[dD][tT][aA]"]     # Drive is searched only if you set DATA_PATH
    hits = [p for pat in pats for p in glob.glob(pat) if "/drive/" not in p]
    pref = [p for p in hits if os.path.basename(p).upper().startswith("BDKR81FL")]
    if pref: return pref[0]
    if hits: return hits[0]
    raise FileNotFoundError("Upload BDKR81FL.DTA to /content (Colab file panel) or set DATA_PATH, then re-run this cell.")

path = find_dta()
need = ["b5", "hw70", "v001", "v005"] + VARS
try:
    raw = pd.read_stata(path, columns=need, convert_categoricals=False)
except (KeyError, ValueError):
    raw = pd.read_stata(path, convert_categoricals=False)
    raw.columns = [c.lower() for c in raw.columns]
    raw = raw[need]
print("loaded", os.path.basename(path), raw.shape)

d = raw[(raw["b5"] == 1) & (raw["hw70"] < 9990)].copy()          # living children with a valid HAZ
y = (d["hw70"] < -200).astype(int).values                         # stunted: HAZ < -2
w = (d["v005"] / 1e6).values                                      # DHS sample weight
groups = d["v001"].astype(int).values                             # cluster (PSU)
Xdf = d[VARS].astype(float).copy()
for v, codes in SPECIAL_CAT_MISSING.items():
    if v in Xdf: Xdf.loc[Xdf[v].isin(codes), v] = np.nan
if "v438" in Xdf: Xdf.loc[Xdf["v438"] >= 9990, "v438"] = np.nan
for v in CAT_VARS: Xdf[v] = Xdf[v].fillna(-1.0)                  # -1 = structural / missing level
X = Xdf.values
print(f"analytic n={len(y)} (earlier pilot: 4105)  stunted={y.sum()} ({y.mean():.3%}; earlier 23.53%)  clusters={len(np.unique(groups))}")
STRICT = os.environ.get("THESIS_STRICT", "1") == "1"
if STRICT and (len(y) != 4105 or int(y.sum()) != 966):
    raise RuntimeError("Cohort differs from the earlier pilot (n=4105, stunted=966). Check the DTA file / filters, or set the environment variable THESIS_STRICT=0 deliberately.")
print("missing share per variable (raw NaN):", {v: round(float(np.isnan(Xdf[v]).mean()), 3) for v in VARS if np.isnan(Xdf[v]).any()} or "none")

## 3. Cluster-grouped, prevalence-stratified outer folds (fresh split)

In [ ]:
sgk = StratifiedGroupKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED_SPLIT)
FOLDS = list(sgk.split(X, y, groups))
for k, (tr, te) in enumerate(FOLDS):
    assert len(set(groups[tr]) & set(groups[te])) == 0, "cluster leakage across train/test"
import hashlib
fold_id = np.zeros(len(y), dtype=np.int8)
for k, (tr, te) in enumerate(FOLDS): fold_id[te] = k
FOLD_SHA = hashlib.sha256(fold_id.tobytes()).hexdigest(); Y_SHA = hashlib.sha256(y.astype(np.int8).tobytes()).hexdigest()
print("fold assignment sha256:", FOLD_SHA[:16], "| outcome vector sha256:", Y_SHA[:16], "| sum of weights:", round(float(w.sum()), 3), "| sklearn", sklearn.__version__)
EXPECTED_FOLD_SHA256 = "8a4f5168872e387ede8307484daf8b360c76abfc3731165f9216ab84ab925667"   # pilot v2 (sklearn 1.6.1)
EXPECTED_Y_SHA256 = "ee4c36ab6cfd2a48ba2a5ee4f9139134fac8ee4e237e582321a0ba0e4c84c410"
if len(y) == 4105 and int(y.sum()) == 966 and os.environ.get("THESIS_ALLOW_VERSION_DRIFT") != "1":
    assert Y_SHA == EXPECTED_Y_SHA256, "outcome vector differs from the pilot (data file or filters changed)"
    assert FOLD_SHA == EXPECTED_FOLD_SHA256, "fold assignment differs from the pilot: scikit-learn version drift in StratifiedGroupKFold? Re-pin scikit-learn==1.6.1 (or record a new hash deliberately)."
print(pd.DataFrame([dict(fold=k, n_train=len(tr), n_test=len(te), test_clusters=len(set(groups[te])),
                         test_prev=round(y[te].mean(), 3)) for k, (tr, te) in enumerate(FOLDS)]).to_string(index=False))

## 4. Preprocessing (fit on the training fold only), models
* Numeric variables: median imputation → standardisation. Categorical variables: one-hot (unknown level → all zeros).
* Explanations are computed at the **source-variable level** (14 players): SHAP and LIME perturb whole variables; IG runs on the encoded inputs and is summed over each variable's dummy columns (valid for IG because the straight-line path moves all dummies of a variable together).
* Explained quantity for every method: predicted probability of the positive class (class-balanced training, so the score is not calibrated to the 23.5% prevalence).

In [ ]:
class Prep:
    def __init__(self, vars_, cat_vars):
        self.vars, self.cat = list(vars_), [v in cat_vars for v in vars_]
    def impute(self, X):
        X = np.array(X, dtype=float, copy=True)
        for j in range(X.shape[1]):
            m = np.isnan(X[:, j])
            if m.any(): X[m, j] = -1.0 if self.cat[j] else self.med[j]
        return X
    def fit(self, X):
        X = np.asarray(X, float)
        self.med = {j: float(np.nanmedian(X[:, j])) for j, c in enumerate(self.cat) if not c}
        Xi = self.impute(X)
        self.mu = {j: Xi[:, j].mean() for j, c in enumerate(self.cat) if not c}
        self.sd = {j: (Xi[:, j].std() or 1.0) for j, c in enumerate(self.cat) if not c}
        self.levels = {j: np.unique(Xi[:, j]) for j, c in enumerate(self.cat) if c}
        self.zcols, self.zvar = [], []
        for j, v in enumerate(self.vars):
            if self.cat[j]:
                for lv in self.levels[j]: self.zcols.append(f"{v}={lv:g}"); self.zvar.append(j)
            else:
                self.zcols.append(v); self.zvar.append(j)
        self.zvar = np.array(self.zvar)
        self.G = np.zeros((len(self.zvar), len(self.vars)))
        self.G[np.arange(len(self.zvar)), self.zvar] = 1.0           # encoded column -> source variable
        return self
    def encode(self, Xi):
        Xi = np.asarray(Xi, float); parts = []
        for j in range(Xi.shape[1]):
            if self.cat[j]: parts.append((Xi[:, [j]] == self.levels[j][None, :]).astype(float))
            else: parts.append(((Xi[:, [j]] - self.mu[j]) / self.sd[j]))
        return np.hstack(parts)

class MLPNet(nn.Module):
    def __init__(self, p, h):
        super().__init__(); self.f = nn.Sequential(nn.Linear(p, h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, z): return self.f(z).squeeze(-1)          # logit

class ProbModule(nn.Module):                                      # probability output, shape (n,), for Captum
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, z): return torch.sigmoid(self.net(z))

class LinLogit(nn.Module):                                        # logistic regression as a float64 torch module (logit output)
    def __init__(self, coef, intercept):
        super().__init__(); self.lin = nn.Linear(coef.shape[1], 1).double()          # float64: the twin reproduces sklearn's float64 LR
        with torch.no_grad():
            self.lin.weight.copy_(torch.as_tensor(coef, dtype=torch.float64)); self.lin.bias.copy_(torch.as_tensor(intercept, dtype=torch.float64))
    def forward(self, z): return self.lin(z).squeeze(-1)

class Fitted:
    def __init__(self, kind, clf=None, net=None): self.kind, self.clf, self.net = kind, clf, net
    def proba(self, Z):
        if self.net is None: return self.clf.predict_proba(Z)[:, 1]
        out = []
        with torch.no_grad():
            for i in range(0, len(Z), 8192):
                out.append(torch.sigmoid(self.net(torch.as_tensor(Z[i:i + 8192], dtype=torch.float32))).numpy())
        return np.concatenate(out).astype(float)
    def prob_module(self):                                          # None for RF (IG not applicable)
        if self.kind == "RF": return None
        net = self.net if self.kind == "MLP" else LinLogit(self.clf.coef_, self.clf.intercept_)   # LR twin: same function as sklearn
        net = net.eval()
        for q in net.parameters(): q.requires_grad_(False)
        return ProbModule(net).eval()

def fit_model(kind, Z, yv, seed):
    if kind == "LR":
        clf = LogisticRegression(C=HP["LR"]["C"], class_weight="balanced", max_iter=5000).fit(Z, yv)
        return Fitted("LR", clf=clf)
    if kind == "RF":
        clf = RandomForestClassifier(class_weight="balanced_subsample", random_state=seed, n_jobs=-1, **HP["RF"]).fit(Z, yv)
        clf.set_params(n_jobs=1)                                    # fit in parallel (tree seeds are independent), PREDICT serially: parallel predict_proba sums trees in a nondeterministic order
        return Fitted("RF", clf=clf)
    h = HP["MLP"]; torch.manual_seed(seed)
    net = MLPNet(Z.shape[1], h["hidden"]); opt = torch.optim.Adam(net.parameters(), lr=h["lr"], weight_decay=h["wd"])
    pw = torch.tensor((1 - yv).sum() / max(yv.sum(), 1), dtype=torch.float32)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    Zt, yt = torch.as_tensor(Z, dtype=torch.float32), torch.as_tensor(yv, dtype=torch.float32)
    gen = torch.Generator().manual_seed(seed)                       # created ONCE: a fresh shuffle every epoch, reproducible from the seed
    for _ in range(h["epochs"]):
        perm = torch.randperm(len(Zt), generator=gen)
        for i in range(0, len(Zt), h["batch"]):
            b = perm[i:i + h["batch"]]; opt.zero_grad(); lossf(net(Zt[b]), yt[b]).backward(); opt.step()
    net.eval()
    return Fitted("MLP", net=net)

def make_predict_raw(prep, fm):
    return lambda Xi: fm.proba(prep.encode(np.asarray(Xi, float)))

## 5. Explainers
*Explainer seed* = the permutation order (SHAP) / the perturbation stream (LIME). The SHAP background (50 training rows) is drawn **once per fold** (`SEED_BG + fold`), so the E-axis floor isolates the explainer's own randomness. `SHAP_bg` is a separate *sensitivity axis*: permutation seed fixed, only the background draw varies (reported as its own floor, never mixed into the primary one). IG is deterministic given the model, so its explainer-seed floor is exactly 1 and only retraining variability exists.

In [ ]:
METHODS_BY_MODEL = dict(LR=["SHAP", "LIME", "IG"], RF=["SHAP", "LIME"], MLP=["SHAP", "LIME", "IG"])

SHAP_PERM_LOG = []                                                  # effective permutations per call (minimum over the explained children)
def shap_perm(f, Xtr_imp, Xin, seed, bg_seed):
    p = Xin.shape[1]
    rng = np.random.RandomState(bg_seed)
    bg = Xtr_imp[rng.choice(len(Xtr_imp), min(B_SHAP, len(Xtr_imp)), replace=False)]
    expl = shap.PermutationExplainer(f, shap.maskers.Independent(bg, max_samples=len(bg)), seed=seed)
    me = N_PERM * (2 * p + 1)
    ex = expl(Xin, max_evals=me, silent=True)                         # shap uses npermutations = max_evals // (2*n_varying + 1); n_varying = variables that differ from the background somewhere
    nvary = (Xin[:, None, :] != bg[None, :, :]).any(1).sum(1)
    SHAP_PERM_LOG.append(int((me // (2 * nvary + 1)).min()))          # == N_PERM unless a variable is constant against the whole background
    vals = np.asarray(ex.values, float)
    add_err = np.abs(vals.sum(1) + np.asarray(ex.base_values, float).reshape(-1) - f(Xin))
    return vals, add_err

def lime_run(f, Xtr_imp, Xin, cat_idx, names, seed):
    expl = lime.lime_tabular.LimeTabularExplainer(
        Xtr_imp, feature_names=names, categorical_features=cat_idx, mode="classification",
        discretize_continuous=True, feature_selection="none", random_state=seed)
    def pf(a):
        p1 = f(a); return np.column_stack([1 - p1, p1])
    out = np.zeros(Xin.shape); r2 = np.zeros(len(Xin))
    for i in range(len(Xin)):
        e = expl.explain_instance(Xin[i], pf, labels=(1,), num_features=Xin.shape[1], num_samples=LIME_SAMPLES)
        for j, wgt in e.local_exp[1]: out[i, j] = wgt
        r2[i] = e.score[1] if isinstance(e.score, dict) else e.score
    return out, r2

def ig_run(fm, prep, Zin, Ztr, Xtr_imp, variants=True):
    pm = fm.prob_module()
    ig = IntegratedGradients(pm)
    dt = torch.float64 if fm.kind == 'LR' else torch.float32          # LR twin is float64; the MLP is a float32 network
    zin = torch.as_tensor(Zin, dtype=dt)
    res, deltas = {}, []
    # baseline A (primary): training mean of the encoded inputs
    b_mean = torch.as_tensor(Ztr.mean(0, keepdims=True), dtype=dt).expand_as(zin)
    a, dl = ig.attribute(zin, baselines=b_mean, n_steps=IG_STEPS, method="gausslegendre", return_convergence_delta=True)
    a = a.numpy().astype(float); deltas.append(np.abs(dl.numpy()))
    res["IG"] = a @ prep.G                                           # signed sum over a variable's dummy columns
    if variants:
        res["IG_sumabs"] = np.abs(a) @ prep.G                       # aggregation sensitivity: sum of |dummy attributions|
        # baseline B (sensitivity): a realisable "typical child" = median numeric / modal category of the training fold
        base_raw = np.array([prep.levels[j][np.bincount(np.searchsorted(prep.levels[j], Xtr_imp[:, j])).argmax()]
                             if prep.cat[j] else prep.med[j] for j in range(Xtr_imp.shape[1])])[None, :]
        b_med = torch.as_tensor(prep.encode(base_raw), dtype=dt).expand_as(zin)
        a2, dl2 = ig.attribute(zin, baselines=b_med, n_steps=IG_STEPS, method="gausslegendre", return_convergence_delta=True)
        res["IG_medmode"] = a2.numpy().astype(float) @ prep.G; deltas.append(np.abs(dl2.numpy()))
    return res, deltas                                                # [primary |delta| array, medmode |delta| array (if variants)]

## 6. Protocol runner (dual-seed design)
For every fold, model and method:
* **E-axis (explainer seed):** model fixed (training seed `TS[0]`), explainer seeds `ES[0..R_E-1]`.
* **T-axis (training seed):** explainer seed fixed (`ES[0]`), model retrained with seeds `TS[0..R_T-1]`.
* Logistic regression has a deterministic solver → no T-axis variation (reported as such, not as "stable").
* Explained children: at most **one child per cluster**, drawn once per fold from the test fold with `SEED_INST`, identical for every model/method/seed.

In [ ]:
def pick_instances(te, groups, n, seed):
    rng = np.random.RandomState(seed); by = {}
    for i in te: by.setdefault(groups[i], []).append(i)
    ids = sorted(by); rng.shuffle(ids)
    return np.array([by[c][rng.randint(len(by[c]))] for c in ids[:n]])

def run_protocol(tag, X, y, w, groups, folds, vars_, cat_vars, R_E, R_T, n_per_fold, ig_variants=True, verbose=True):
    cat_idx = [j for j, v in enumerate(vars_) if v in cat_vars]
    out = dict(attr={}, perf=[], inst=[], diag=[], timers={}, tag=tag, vars=list(vars_), n_expl=0)
    T = out["timers"]
    def tick(key, t0, n=1):
        a = T.setdefault(key, [0.0, 0]); a[0] += time.time() - t0; a[1] += n
    for k, (tr, te) in enumerate(folds):
        t_fold = time.time()
        prep = Prep(vars_, cat_vars).fit(X[tr])
        Xtr = prep.impute(X[tr]); Ztr = prep.encode(Xtr); ytr = y[tr]
        Zte = prep.encode(prep.impute(X[te]))
        sel = pick_instances(te, groups, n_per_fold, SEED_INST + k)
        Xin = prep.impute(X[sel]); Zin = prep.encode(Xin)
        inst = pd.DataFrame(dict(tag=tag, fold=k, inst=range(len(sel)), y=y[sel]))
        for m in ["LR", "RF", "MLP"]:
            n_ts = 1 if m == "LR" else R_T
            fits = []
            for r in range(n_ts):
                t0 = time.time(); fits.append(fit_model(m, Ztr, ytr, TS[r])); tick((m, "fit"), t0)
                pte = fits[-1].proba(Zte)
                out["perf"].append(dict(tag=tag, fold=k, model=m, train_seed_idx=r,
                                        auc_w=roc_auc_score(y[te], pte, sample_weight=w[te]), auc_u=roc_auc_score(y[te], pte)))
            inst[f"p_{m}"] = fits[0].proba(Zin)
            if m == "LR":                                           # twin check: PyTorch LR == sklearn LR
                twin = fits[0].prob_module()(torch.as_tensor(Zin, dtype=torch.float64)).detach().numpy()
                out["diag"].append(dict(tag=tag, fold=k, model=m, check="lr_twin_max_abs_diff", value=float(np.abs(twin - inst["p_LR"].values).max())))
            fs = [make_predict_raw(prep, f) for f in fits]
            for meth in METHODS_BY_MODEL[m]:
                if meth in ("SHAP", "LIME"):
                    def run_once(f, seed, bg_seed=None):
                        t0 = time.time()
                        if meth == "SHAP":
                            a, err = shap_perm(f, Xtr, Xin, seed, SEED_BG + k if bg_seed is None else bg_seed)
                            out["diag"].append(dict(tag=tag, fold=k, model=m, check="shap_additivity_max_err", value=float(err.max())))
                        else:
                            a, r2 = lime_run(f, Xtr, Xin, cat_idx, list(vars_), seed)
                            out["diag"].append(dict(tag=tag, fold=k, model=m, check="lime_local_r2_mean", value=float(r2.mean())))
                        tick((m, meth), t0, len(Xin)); out["n_expl"] += len(Xin)
                        return a
                    for r in range(R_E):
                        out["attr"][(k, m, meth, "E", r)] = run_once(fs[0], ES[r])
                    out["attr"][(k, m, meth, "T", 0)] = out["attr"][(k, m, meth, "E", 0)]
                    for r in range(1, n_ts):
                        out["attr"][(k, m, meth, "T", r)] = run_once(fs[r], ES[0])
                    if meth == "SHAP" and R_E > 1:                      # sensitivity axis: only the background draw varies
                        for r in range(R_E):
                            out["attr"][(k, m, "SHAP_bg", "E", r)] = run_once(fs[0], ES[0], bg_seed=ES[r])
                else:                                               # IG family (deterministic given the model)
                    for r in range(n_ts):
                        t0 = time.time()
                        res, dls = ig_run(fits[r], prep, Zin, Ztr, Xtr, variants=ig_variants)
                        tick((m, "IG"), t0, len(Xin)); out["n_expl"] += len(Xin)
                        out["diag"].append(dict(tag=tag, fold=k, model=m, check="ig_delta_max", run=r, value=float(dls[0].max())))
                        if len(dls) > 1: out["diag"].append(dict(tag=tag, fold=k, model=m, check="ig_medmode_delta_max", run=r, value=float(dls[1].max())))
                        for name, arr in res.items():
                            out["attr"][(k, m, name, "T", r)] = arr
                            if r == 0: out["attr"][(k, m, name, "E", 0)] = arr
        out["inst"].append(inst)
        if verbose: print(f"[{tag}] fold {k} done in {time.time() - t_fold:5.0f}s  (explained {len(sel)} children)")
    out["perf"] = pd.DataFrame(out["perf"]); out["inst"] = pd.concat(out["inst"], ignore_index=True); out["diag"] = pd.DataFrame(out["diag"])
    return out

## 6a. Agreement metrics (defined here so they can be unit-tested; used in section 9)

In [ ]:
def _top(v, k): return set(np.argsort(-np.abs(v), kind="stable")[:k])
def sim_tv(a, b):
    pa, pb = np.abs(a), np.abs(b); sa, sb = pa.sum(), pb.sum()
    pa = pa / sa if sa > 0 else np.full_like(pa, 1 / len(pa)); pb = pb / sb if sb > 0 else np.full_like(pb, 1 / len(pb))
    return 1.0 - 0.5 * np.abs(pa - pb).sum()
def sim_spearman(a, b):
    ra, rb = rankdata(np.abs(a)), rankdata(np.abs(b))
    if ra.std() == 0 or rb.std() == 0: return 0.0
    return float(np.corrcoef(ra, rb)[0, 1])
def sim_jaccard(a, b, k=TOPK):
    A, B = _top(a, k), _top(b, k); return len(A & B) / len(A | B)
def sim_rbo(a, b, p=0.9):
    oa, ob = np.argsort(-np.abs(a), kind="stable"), np.argsort(-np.abs(b), kind="stable"); D = len(a)
    wts = p ** np.arange(D); ag = [len(set(oa[:d]) & set(ob[:d])) / d for d in range(1, D + 1)]
    return float((wts * np.array(ag)).sum() / wts.sum())
METRICS = {"1-TV": sim_tv, "Spearman": sim_spearman, f"Jaccard@{TOPK}": sim_jaccard, "RBO": sim_rbo}

EPS = 1e-12
def kappa_parts(mC, mA, mB, mN):
    """(ceiling, kappa, excess disagreement). kappa is NaN (undefined, never inf) when 1-N ~ 0 or when the ceiling is at the null (a floor is at/below the null)."""
    mC, mA, mB, mN = (np.asarray(v, float) for v in (mC, mA, mB, mN))
    with np.errstate(divide="ignore", invalid="ignore"):
        d1 = 1 - mN
        rA = np.where(np.abs(d1) > EPS, np.clip((mA - mN) / d1, 0, 1), np.nan)
        rB = np.where(np.abs(d1) > EPS, np.clip((mB - mN) / d1, 0, 1), np.nan)
        ceil = mN + d1 * np.sqrt(rA * rB)
        d2 = ceil - mN
        kap = np.where(np.abs(d2) > EPS, (mC - mN) / d2, np.nan)
    return ceil, kap, ceil - mC
def cons_floor(FE, FT):
    """Conservative per-child floor = min(E-axis floor, T-axis floor) child by child where a T-axis floor exists; E-axis only otherwise (LR)."""
    FE, FT = np.asarray(FE, float), np.asarray(FT, float)
    return np.where(np.isnan(FT), FE, np.minimum(FE, FT))
def kappa_status(mN, kap):
    return "ok" if np.isfinite(kap) else ("UNDEFINED: null ~ 1" if abs(1 - mN) <= EPS else "UNDEFINED: a floor is at/below the null")

## 6b. Stage gate 0 — unit tests (seconds)
Metric sanity, seed behaviour of the explainers, and no leakage in the encoding. A failure here means a code problem, not a scientific result.

In [ ]:
UNIT_TESTS_PASSED = False; _t_unit = time.time()
_a = np.array([5., 4, 3, 2, 1, 0.5, 0.4, 0.3, 0.2, 0.1, 0.05, 0.04, 0.03, 0.02]); _b = _a[::-1].copy()
assert abs(sim_tv(_a, _a) - 1) < 1e-12 and abs(sim_spearman(_a, _a) - 1) < 1e-12 and sim_jaccard(_a, _a) == 1 and abs(sim_rbo(_a, _a) - 1) < 1e-12
assert abs(sim_spearman(_a, _b) + 1) < 1e-12 and sim_jaccard(_a, _b) == 0 and sim_rbo(_a, _b) < 0.25 and sim_tv(_a, _b) < 1
assert abs(sim_tv(-_a, _a) - 1) < 1e-12, "importance must use |attribution|"
_tr, _te = FOLDS[0]; _pr = Prep(VARS, CAT_VARS).fit(X[_tr]); _Xtr = _pr.impute(X[_tr]); _Ztr = _pr.encode(_Xtr)
assert _Ztr.shape[1] == len(_pr.zcols) == _pr.G.shape[0] and np.allclose(_pr.G.sum(1), 1) and not np.isnan(_Ztr).any()
assert np.allclose(_Ztr[:, [j for j, c in enumerate(_pr.zcols) if "=" not in c]].mean(0), 0, atol=1e-8), "numeric columns must be standardised on the training fold"
_sel = pick_instances(_te, groups, 4, SEED_INST); assert len(set(groups[_sel])) == len(_sel), "more than one child per cluster"
_Xin = _pr.impute(X[_sel]); _fm = fit_model("MLP", _Ztr, y[_tr], TS[0]); _f = make_predict_raw(_pr, _fm)
_s1, _ = shap_perm(_f, _Xtr, _Xin, ES[0], SEED_BG); _s2, _ = shap_perm(_f, _Xtr, _Xin, ES[0], SEED_BG); _s3, _ = shap_perm(_f, _Xtr, _Xin, ES[1], SEED_BG); _s4, _ = shap_perm(_f, _Xtr, _Xin, ES[0], SEED_BG + 1)
assert np.allclose(_s1, _s2) and not np.allclose(_s1, _s3) and not np.allclose(_s1, _s4), "SHAP: same (seed, background) must reproduce; a different seed OR a different background must change the result"
_l1, _ = lime_run(_f, _Xtr, _Xin, [j for j, v in enumerate(VARS) if v in CAT_VARS], VARS, ES[0]); _l2, _ = lime_run(_f, _Xtr, _Xin, [j for j, v in enumerate(VARS) if v in CAT_VARS], VARS, ES[0])
_l3, _ = lime_run(_f, _Xtr, _Xin, [j for j, v in enumerate(VARS) if v in CAT_VARS], VARS, ES[1])
assert np.allclose(_l1, _l2) and not np.allclose(_l1, _l3), "LIME: same seed must reproduce, different seed must differ"
_i1, _ = ig_run(_fm, _pr, _pr.encode(_Xin), _Ztr, _Xtr); _i2, _ = ig_run(_fm, _pr, _pr.encode(_Xin), _Ztr, _Xtr)
assert np.allclose(_i1["IG"], _i2["IG"]), "IG must be deterministic"
# kappa / conservative-floor logic (the auditor's counter-example and the undefined-denominator cases)
_FE, _FT = np.array([.9, .1]), np.array([.1, .9])
assert abs(cons_floor(_FE, _FT).mean() - 0.1) < 1e-12 and abs(min(_FE.mean(), _FT.mean()) - 0.5) < 1e-12, "conservative floor must be the per-child minimum, not the minimum of means"
assert np.allclose(cons_floor(_FE, np.full(2, np.nan)), _FE), "no T-axis (LR): E-axis floor only"
_c, _k, _d = kappa_parts(0.7, 0.9, 0.9, 0.2); assert abs(float(_c) - 0.9) < 1e-12 and abs(float(_k) - 0.5 / 0.7) < 1e-12 and abs(float(_d) - 0.2) < 1e-12
assert np.isnan(kappa_parts(0.7, 0.9, 0.9, 1.0)[1]), "N2 == 1 must give an undefined kappa, never inf"
assert np.isnan(kappa_parts(0.7, 0.1, 0.9, 0.2)[1]), "a floor at/below the null must give an undefined kappa"
assert fit_model("RF", _Ztr, y[_tr], TS[0]).clf.n_jobs == 1, "RF must predict serially"
WORK["unit_tests"] = time.time() - _t_unit; UNIT_TESTS_PASSED = True
print("unit tests passed")

## 7. Stage gate 1 — wiring + compute projection (≈ 1–3 min)
Runs fold 0 with 5 children and one run per axis, times each method, and projects the cost of the configured MODE.

In [ ]:
def runs_per_child(m, meth, R_E, R_T):
    n_ts = 1 if m == "LR" else R_T
    if meth == "SHAP": return R_E + n_ts - 1 + (R_E if R_E > 1 else 0)   # + the background-sensitivity axis
    if meth == "LIME": return R_E + n_ts - 1
    return n_ts                                                      # IG: one call-set per fitted model

t_probe = time.time()
probe = run_protocol("probe", X, y, w, groups, FOLDS[:1], VARS, CAT_VARS, R_E=1, R_T=1, n_per_fold=5, verbose=False)
proj_sec = 0.0
for (m, key), (sec, n) in probe["timers"].items():
    if key == "fit": proj_sec += sec * (1 if m == "LR" else R_T) * N_OUTER_FOLDS
    else: proj_sec += (sec / n) * runs_per_child(m, key, R_E, R_T) * N_PER_FOLD * N_OUTER_FOLDS
proj_sec = proj_sec * 1.25 + (time.time() - t_probe)                 # + negative-control run, metrics, overhead, and the probe itself
WORK['probe'] = time.time() - t_probe
proj_h = proj_sec / 3600
print(f"probe took {time.time() - t_probe:.0f}s | projected run time for MODE={MODE}: {proj_h:.2f} h (gate: <= {GATES['max_projected_hours']} h)")
print({f"{m}/{k}": (f"{sec / n:.3f}s per child" if k != "fit" else f"{sec:.1f}s per fit") for (m, k), (sec, n) in probe["timers"].items()})
assert proj_h <= GATES["max_projected_hours"], "Projected run time exceeds the pre-registered budget: reduce N_PER_FOLD (or fix a slow component) BEFORE running."

## 8. MAIN RUN

In [ ]:
t_main = time.time()
MAIN = run_protocol("main", X, y, w, groups, FOLDS, VARS, CAT_VARS, R_E=R_E, R_T=R_T, n_per_fold=N_PER_FOLD)
WORK["main"] = time.time() - t_main
print(f"main run finished in {(time.time() - t_main) / 60:.1f} min; total explanations: {MAIN['n_expl']}")
perf = MAIN["perf"]
print(perf[perf.train_seed_idx == 0].groupby("model")[["auc_w", "auc_u"]].agg(["mean", "std"]).round(3))

## 9. Agreement metrics, noise floor, null calibration
All metrics are computed per child on the **14 variable-level importances** `|attribution|` and are expressed as *similarities* (higher = more agreement):
`1-TV` (magnitude), Spearman (rank), top-5 Jaccard (set), and RBO@14 with p=0.9: Σ_d p^(d−1)·A_d / Σ_d p^(d−1), where A_d is the overlap of the two top-d lists divided by d. It is **normalised over the 14 available ranks** (identical rankings score exactly 1), which is *not* the extrapolated RBO of Webber et al.; state it that way in any write-up.

* **Floor** of method A = mean self-similarity of A's own repeated runs (E-axis: explainer seeds; T-axis: retrained models).
* **Cross** similarity C(A,B) = mean over all run pairs (A_r, B_s) on the same model and children.
* **Null N1** = similarity of two random importance vectors (Dirichlet(1)). **Null N2** = similarity of A(child i) with B(a *different* child j) — keeps the global importance profile, destroys child-level alignment. N2 is the stricter reference.
* **Ceiling and κ.** Noise in A and B limits how similar they can look even if they agreed perfectly. On the null-adjusted scale, reliability ρ = (F − N)/(1 − N) (clipped to [0,1]); ceiling = N + (1 − N)·√(ρ_A·ρ_B) (the usual attenuation correction, exact for correlation-type metrics, a heuristic for the others).
  **κ = (C − N)/(ceiling − N)**: 1 means "as similar as the two explainers' own noise allows"; 0 means "no better than the null". **Excess disagreement** Δ = ceiling − C, in similarity units. A deterministic explainer (IG) has ρ = 1 on the explainer-seed axis.
* The *conservative* κ uses, for each method, the **per-child minimum** of its explainer-seed and retraining floors (E only for LR, which has no retraining floor); gate G7 uses the same quantity. κ is a **ceiling ratio, not a bounded agreement coefficient**: values outside [0, 1] are possible and are flagged (`kappa_outside_0_1`), never clipped.
* Bootstrap intervals resample the selected children only. They are **conditional on the fitted folds and models** and are therefore narrower than the true uncertainty; `kappa_by_fold.csv` shows the between-fold spread.
* Ties: ranks use average ties (Spearman) and column-order tie-breaking (top-k, RBO). `tie_diagnostics.csv` and gate G8 check that this cannot matter (all-zero vectors and ties at the top-k boundary must be rare).

In [ ]:
def sim_vec(A, B, fn): return np.array([fn(A[i], B[i]) for i in range(len(A))])
def mean_pairs(runsA, runsB, fn, same):
    vals = []
    for r, s in itertools.product(range(len(runsA)), range(len(runsB))):
        if same and s <= r: continue
        vals.append(sim_vec(runsA[r], runsB[s], fn))
    return np.mean(vals, axis=0) if vals else None
def get_runs(out, k, m, meth, axis):
    ks = sorted(r for (kk, mm, me, ax, r) in out["attr"] if (kk, mm, me, ax) == (k, m, meth, axis))
    return [out["attr"][(k, m, meth, axis, r)] for r in ks]

def null_n1(nvars, n_pairs=20000, seed=SEED_NULL):
    rng = np.random.RandomState(seed); P = rng.dirichlet(np.ones(nvars), size=(n_pairs, 2))
    return {name: float(np.mean([fn(P[i, 0], P[i, 1]) for i in range(n_pairs)])) for name, fn in METRICS.items()}
N1 = null_n1(len(VARS)); print("N1 (random-importance null):", {k: round(v, 3) for k, v in N1.items()})

def n2_child(A0, B0, fn):                                           # A(child i) vs B(every other child j) within the fold, symmetrised over both directions
    n = len(A0)
    if n < 2: return np.full(n, np.nan)
    return np.array([0.5 * np.mean([fn(A0[i], B0[j]) + fn(A0[j], B0[i]) for j in range(n) if j != i]) for i in range(n)])

def collect(out, models=("LR", "RF", "MLP")):
    """Per-child arrays pooled over folds: floors (E,T), cross, N2."""
    rows = {}
    nf = len({k for (k, *_ ) in out["attr"]})
    for m in models:
        for meth in METHODS_BY_MODEL[m] + (["IG_sumabs", "IG_medmode"] if "IG" in METHODS_BY_MODEL[m] else []) + (["SHAP_bg"] if any(key[1:4] == (m, "SHAP_bg", "E") for key in out["attr"]) else []):
            for fn_name, fn in METRICS.items():
                fE, fT = [], []
                for k in range(nf):
                    E, Tt = get_runs(out, k, m, meth, "E"), get_runs(out, k, m, meth, "T")
                    n = len(E[0])
                    fE.append(mean_pairs(E, E, fn, True) if len(E) > 1 else np.ones(n))      # deterministic -> perfectly reproducible
                    fT.append(mean_pairs(Tt, Tt, fn, True) if len(Tt) > 1 else np.full(n, np.nan))
                rows[(m, meth, fn_name)] = dict(FE=np.concatenate(fE), FT=np.concatenate(fT), fold=np.concatenate([np.full(len(x), k) for k, x in enumerate(fE)]))
    cross = {}
    for m in models:
        meths = [x for x in METHODS_BY_MODEL[m]]
        for A, B in itertools.combinations(meths, 2):
            for fn_name, fn in METRICS.items():
                C, N2 = [], []
                for k in range(nf):
                    EA, EB = get_runs(out, k, m, A, "E"), get_runs(out, k, m, B, "E")
                    C.append(mean_pairs(EA, EB, fn, False)); N2.append(n2_child(EA[0], EB[0], fn))
                cross[(m, A, B, fn_name)] = dict(C=np.concatenate(C), N2=np.concatenate(N2), fold=np.concatenate([np.full(len(x), k) for k, x in enumerate(C)]))
    return rows, cross

def boot_stats(C, F_A, F_B, N, B=2000, seed=SEED_BOOT):
    n = len(C); rng = np.random.RandomState(seed); idx = rng.randint(0, n, (B, n))
    _, kap, dlt = kappa_parts(C[idx].mean(1), F_A[idx].mean(1), F_B[idx].mean(1), N[idx].mean(1))
    q = lambda v: (float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5)))
    return q(kap), q(dlt)

FLOOR, CROSS = collect(MAIN)
rows = []
for (m, A, B, fn_name), c in CROSS.items():
    fa, fb = FLOOR[(m, A, fn_name)], FLOOR[(m, B, fn_name)]
    ok = ~np.isnan(c["N2"]); C, N2 = c["C"][ok], c["N2"][ok]
    FEa, FEb, FTa, FTb = fa["FE"][ok], fb["FE"][ok], fa["FT"][ok], fb["FT"][ok]
    mC, mN2, mN1 = C.mean(), N2.mean(), N1[fn_name]
    CFa, CFb = cons_floor(FEa, FTa), cons_floor(FEb, FTb)
    ceil2, kap2, dlt2 = kappa_parts(mC, FEa.mean(), FEb.mean(), mN2)
    _, kap1, _ = kappa_parts(mC, FEa.mean(), FEb.mean(), mN1)
    _, kapc, _ = kappa_parts(mC, CFa.mean(), CFb.mean(), mN2)
    k2, d = boot_stats(C, FEa, FEb, N2)
    rows.append({"model": m, "pair": f"{A} vs {B}", "metric": fn_name, "n_children": int(ok.sum()),
                 "cross_C": mC, "floorE_A": FEa.mean(), "floorE_B": FEb.mean(),
                 "floorT_A": np.nan if np.isnan(FTa).all() else np.nanmean(FTa), "floorT_B": np.nan if np.isnan(FTb).all() else np.nanmean(FTb),
                 "null_N1": mN1, "null_N2": mN2, "ceiling_N2": float(ceil2),
                 "excess_disagreement": float(dlt2), "excess_CI95": d,
                 "kappa_N2": float(kap2), "kappa_status": kappa_status(mN2, float(kap2)), "kappa_outside_0_1": bool(np.isfinite(kap2) and not (0 <= float(kap2) <= 1)), "kappa_N2_CI95": k2, "kappa_N1": float(kap1), "kappa_conservative_N2": float(kapc), "kappa_conservative_outside_0_1": bool(np.isfinite(kapc) and not (0 <= float(kapc) <= 1)),
                 "min_floor_minus_N2": min(CFa.mean(), CFb.mean()) - mN2, "min_floor_minus_N2_Eonly": min(FEa.mean(), FEb.mean()) - mN2})
CROSS_TABLE = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 40)
print(CROSS_TABLE[CROSS_TABLE.metric == "Spearman"][["model", "pair", "cross_C", "floorE_A", "floorE_B", "null_N2", "ceiling_N2", "excess_disagreement",
                                                    "kappa_N2", "kappa_N2_CI95", "kappa_conservative_N2"]].round(3).to_string(index=False))

# Tie / all-zero diagnostics: tie-breaking by column order must not be able to drive the rank metrics.
rows = []
for (k, m, meth, ax_, r), arr in MAIN["attr"].items():
    A = np.abs(arr); srt = -np.sort(-A, axis=1)
    rows.append(dict(model=m, method=meth, n_vectors=len(A), all_zero=int((A.sum(1) == 0).sum()),
                     tie_at_topk_boundary=int((np.isclose(srt[:, TOPK - 1], srt[:, TOPK], rtol=0, atol=1e-12) & (A.sum(1) > 0)).sum())))
TIE_DIAG = pd.DataFrame(rows).groupby(["model", "method"], as_index=False).sum(numeric_only=True)
TIE_DIAG["tie_share"] = (TIE_DIAG.all_zero + TIE_DIAG.tie_at_topk_boundary) / TIE_DIAG.n_vectors
print(TIE_DIAG.round(4).to_string(index=False))

# Between-fold spread of kappa (the bootstrap interval above is conditional on the fitted folds and selected children; this shows how much the fitted models matter)
rows = []
for (m, A, B, fn_name), c in CROSS.items():
    fa, fb = FLOOR[(m, A, fn_name)], FLOOR[(m, B, fn_name)]
    for k in range(N_OUTER_FOLDS):
        i = (c["fold"] == k) & ~np.isnan(c["N2"])
        if i.sum() < 2: continue
        _, kap, dlt = kappa_parts(c["C"][i].mean(), fa["FE"][i].mean(), fb["FE"][i].mean(), c["N2"][i].mean())
        rows.append(dict(model=m, pair=f"{A} vs {B}", metric=fn_name, fold=k, n_children=int(i.sum()), kappa_N2=float(kap), excess_disagreement=float(dlt)))
KAPPA_BY_FOLD = pd.DataFrame(rows)
print(KAPPA_BY_FOLD[KAPPA_BY_FOLD.metric == "Spearman"].pivot_table(index=["model", "pair"], columns="fold", values="kappa_N2").round(2).to_string())

### Floors (reproducibility of each explainer on its own) and IG variants

In [ ]:
rows = []
for (m, meth, fn_name), v in FLOOR.items():
    rows.append(dict(model=m, method=meth, metric=fn_name, floor_explainer_seed=v["FE"].mean(),
                     floor_retrain_seed=(np.nan if np.isnan(v["FT"]).all() else np.nanmean(v["FT"])),
                     note="; ".join(x for x in ["deterministic explainer" if meth.startswith("IG") else "", "LR solver deterministic (no retrain variation)" if m == "LR" else ""] if x)))
FLOOR_TABLE = pd.DataFrame(rows)
print(FLOOR_TABLE[(FLOOR_TABLE.metric == "Spearman") & (~FLOOR_TABLE.method.isin(["IG_sumabs", "IG_medmode"]))].round(3).to_string(index=False))

# IG sensitivity (baseline / aggregation): similarity of the variant to the primary IG, same model, T-axis run 0
rows = []
for m in ["LR", "MLP"]:
    for var in ["IG_medmode", "IG_sumabs"]:
        for fn_name, fn in METRICS.items():
            v = np.concatenate([sim_vec(MAIN["attr"][(k, m, "IG", "T", 0)], MAIN["attr"][(k, m, var, "T", 0)], fn) for k in range(N_OUTER_FOLDS)])
            rows.append(dict(model=m, variant=var, metric=fn_name, similarity_to_primary_IG=v.mean()))
IG_VARIANTS = pd.DataFrame(rows); print(IG_VARIANTS.round(3).to_string(index=False))

## 10. Same method, different model types (secondary)
Not a floor-adjusted quantity: the models differ, so part of this gap is real. Compare with each model's retrain floor (table above).

In [ ]:
rows = []
for meth in ["SHAP", "LIME", "IG"]:
    ms = [m for m in ["LR", "RF", "MLP"] if meth in METHODS_BY_MODEL[m]]
    for ma, mb in itertools.combinations(ms, 2):
        for fn_name, fn in METRICS.items():
            v = np.concatenate([sim_vec(MAIN["attr"][(k, ma, meth, "E", 0)], MAIN["attr"][(k, mb, meth, "E", 0)], fn) for k in range(N_OUTER_FOLDS)])
            rows.append(dict(method=meth, models=f"{ma} vs {mb}", metric=fn_name, similarity=v.mean()))
CROSS_MODEL = pd.DataFrame(rows)
print(CROSS_MODEL[CROSS_MODEL.metric == "Spearman"].round(3).to_string(index=False))

## 11. Stage gate 2 — negative-control column
A pre-generated N(0,1) column is appended as a 15th variable. Every explainer should rank it near the bottom. One run per method (E/T = first seed), 20 children per fold.

In [ ]:
rng_noise = np.random.RandomState(SEED_NOISE)
Xn = np.column_stack([X, rng_noise.randn(len(X))]); vars_n = VARS + ["random_noise"]
_t_ctrl = time.time()
CTRL = run_protocol("noise_control", Xn, y, w, groups, FOLDS, vars_n, CAT_VARS, R_E=1, R_T=1,
                    n_per_fold=min(20, N_PER_FOLD) if MODE != "SMOKE" else N_PER_FOLD, ig_variants=False)
WORK["noise_control"] = time.time() - _t_ctrl
rows = []
for m in ["LR", "RF", "MLP"]:
    for meth in METHODS_BY_MODEL[m]:
        ranks, shares, real = [], [], []
        for k in range(N_OUTER_FOLDS):
            A = np.abs(CTRL["attr"][(k, m, meth, "E", 0)])
            ranks.extend((rankdata(-A, axis=1)[:, -1]).tolist()); shares.extend((A[:, -1] / A.sum(1)).tolist()); real.append((A[:, :-1] / A.sum(1, keepdims=True)).mean(0))
        med_real = float(np.median(np.mean(real, axis=0)))               # informational: how does the noise column compare with a TYPICAL real variable (many real variables are weak too)?
        rows.append(dict(model=m, method=meth, median_rank_of_noise=float(np.median(ranks)), mean_importance_share=float(np.mean(shares)), n=len(ranks), uniform_share=1 / len(vars_n),
                         median_real_variable_share=med_real, noise_over_median_real=float(np.mean(shares)) / med_real))
NOISE_TABLE = pd.DataFrame(rows); print(NOISE_TABLE.round(3).to_string(index=False))

## 12. Optional consistency check — exact TreeSHAP (RF) vs variable-level Permutation SHAP
TreeSHAP runs on the encoded inputs and is summed over each variable's dummies. For one-hot groups the sum of per-dummy Shapley values is **not** the Shapley value of the grouped game, so perfect agreement is not expected; a very low value would signal a bug.

In [ ]:
RUN_REFERENCE_CHECK = True
REF = None
if RUN_REFERENCE_CHECK:
    _t_ref = time.time()
    tr, te = FOLDS[0]
    prep = Prep(VARS, CAT_VARS).fit(X[tr]); Xtr = prep.impute(X[tr]); Ztr = prep.encode(Xtr)
    sel = pick_instances(te, groups, N_PER_FOLD, SEED_INST + 0)
    Zin = prep.encode(prep.impute(X[sel]))
    rf = fit_model("RF", Ztr, y[tr], TS[0])
    bg = Ztr[np.random.RandomState(ES[0]).choice(len(Ztr), 100, replace=False)]
    sv = shap.TreeExplainer(rf.clf, data=bg, feature_perturbation="interventional", model_output="probability").shap_values(Zin, check_additivity=False)
    sv = np.asarray(sv[1] if isinstance(sv, list) else (sv[..., 1] if np.ndim(sv) == 3 else sv))
    tree_var = sv @ prep.G
    perm_mean = np.mean([MAIN["attr"][(0, "RF", "SHAP", "E", r)] for r in range(R_E)], axis=0)
    REF = dict(spearman_tree_vs_perm=float(np.mean([sim_spearman(tree_var[i], perm_mean[i]) for i in range(len(sel))])),
               jaccard_tree_vs_perm=float(np.mean([sim_jaccard(tree_var[i], perm_mean[i]) for i in range(len(sel))])))
    print(REF); WORK['reference_check'] = time.time() - _t_ref

## 13. Gate report

In [ ]:
G = GATES; gate = []
def add(name, ok, detail): gate.append(dict(gate=name, passed=bool(ok), detail=detail))
add("G0 unit tests (cell 6b) ran and passed in this session", UNIT_TESTS_PASSED, "assertions passed before the run" if UNIT_TESTS_PASSED else "cell 6b did not complete in this session")
work_h = sum(WORK.values()) / 3600
add("G1 compute budget (projection incl. probe AND measured work: probe, main run, controls, TreeSHAP check)", proj_h <= G["max_projected_hours"] and work_h <= G["max_projected_hours"], f"projected {proj_h:.2f} h; measured {work_h:.2f} h {({k: round(v / 60, 1) for k, v in WORK.items()})} min")
auc = MAIN["perf"][MAIN["perf"].train_seed_idx == 0].groupby("model").auc_w.mean()
add("G2 models above trivial", len(auc) == 3 and bool((auc >= G["min_auc_weighted"]).all()), auc.round(3).to_dict())
gap = {m: round(abs(auc[m] - EARLIER_S0_AUC[m]), 3) for m in auc.index}
add("G2b close to earlier S0 AUROC (new folds -> small differences expected)", all(v <= G["max_auc_gap_vs_earlier"] for v in gap.values()) if FEATURE_SET == "S0" else True, gap)
add("G3 null calibration (Spearman N1 ~ 0)", abs(N1["Spearman"]) <= G["null_spearman_abs"], round(N1["Spearman"], 4))
dg = MAIN["diag"]
shap_err = dg[dg.check == "shap_additivity_max_err"].value.max(); ig_d = dg[dg.check.isin(["ig_delta_max", "ig_medmode_delta_max"])].value.max()
add("G4a SHAP additivity", shap_err <= G["max_additivity_err"], f"max err {shap_err:.2e}")
add("G4b IG completeness", ig_d <= G["max_ig_delta"], f"max |delta| {ig_d:.2e}")
twin = dg[dg.check == "lr_twin_max_abs_diff"].value.max()
add("G4c PyTorch LR twin equals sklearn LR", twin < 1e-5, f"max diff {twin:.2e}")
n_exp_noise = sum(len(v) for v in METHODS_BY_MODEL.values())
add("G5 noise column ranks near bottom (a failure means the FITTED MODEL leans on noise; it is not by itself an explainer bug)", len(NOISE_TABLE) == n_exp_noise and bool(((NOISE_TABLE.median_rank_of_noise >= G["noise_min_median_rank"]) & (NOISE_TABLE.mean_importance_share <= G["noise_max_share_ratio"] * NOISE_TABLE.uniform_share)).all()),
    {f"{r.model}/{r.method}": (round(r.median_rank_of_noise, 1), round(r.mean_importance_share / r.uniform_share, 2)) for r in NOISE_TABLE.itertuples()})
sl = CROSS_TABLE[(CROSS_TABLE.metric == "Spearman") & (CROSS_TABLE.pair == "SHAP vs LIME")]
width = {r.model: round(r.kappa_N2_CI95[1] - r.kappa_N2_CI95[0], 3) for r in sl.itertuples()}
add("G6 kappa precision (SHAP vs LIME, Spearman; interval is CONDITIONAL on the selected children and fitted folds; undefined kappa fails)", len(width) == 3 and all(np.isfinite(v) and v <= G["max_kappa_ci_width"] for v in width.values()) and bool((sl.kappa_status == "ok").all()), width)
add("G7 per-child-minimum floors (E and T) exceed the null (kappa stable)", len(sl) == 3 and bool((sl.min_floor_minus_N2 >= G["min_floor_minus_null"]).all()), sl.set_index("model").min_floor_minus_N2.round(3).to_dict())
add("G8 ties / all-zero attribution vectors", bool(TIE_DIAG.tie_share.max() <= G["max_tie_share"]), f"max share {TIE_DIAG.tie_share.max():.4f}")
lime_r2 = dg[dg.check == "lime_local_r2_mean"].groupby("model").value.mean().round(3).to_dict()
GATE = pd.DataFrame(gate); print(GATE.to_string(index=False))
print("\nLIME mean local R^2 (surrogate fidelity, informational):", lime_r2)
print("SHAP effective permutations per child (minimum over all calls; pre-registered N_PERM =", N_PERM, "):", min(SHAP_PERM_LOG) if SHAP_PERM_LOG else "n/a")
print("Noise column vs a typical real variable (informational; G5 is judged on its pre-registered thresholds only):")
print(NOISE_TABLE[["model", "method", "median_rank_of_noise", "noise_over_median_real"]].round(2).to_string(index=False))
print("\nOVERALL:", "PASS - run the same code unchanged with MODE='FULL'" if GATE.passed.all() else "FAIL - fix the failing gate(s) as a *protocol* issue; do not change settings to improve results.")

## 14. Figures

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sp = CROSS_TABLE[CROSS_TABLE.metric == "Spearman"].copy(); sp["label"] = sp.model + "\n" + sp.pair
x = np.arange(len(sp)); ax[0].bar(x - 0.27, sp.ceiling_N2, 0.27, label="ceiling (noise-limited)")
ax[0].bar(x, sp.cross_C, 0.27, label="cross-method C"); ax[0].bar(x + 0.27, sp.null_N2, 0.27, label="null N2")
ax[0].set_xticks(x); ax[0].set_xticklabels(sp.label, fontsize=7); ax[0].set_ylabel("Spearman similarity"); ax[0].legend(fontsize=7); ax[0].set_title("Ceiling vs cross-method vs null")
lo = sp.kappa_N2 - sp.kappa_N2_CI95.map(lambda t: t[0]); hi = sp.kappa_N2_CI95.map(lambda t: t[1]) - sp.kappa_N2
ax[1].errorbar(x, sp.kappa_N2, yerr=[lo, hi], fmt="o"); ax[1].axhline(1, ls="--", c="gray"); ax[1].axhline(0, ls=":", c="gray")
ax[1].set_xticks(x); ax[1].set_xticklabels(sp.label, fontsize=7); ax[1].set_ylabel("kappa (N2 null)"); ax[1].set_title("Floor-normalised agreement (1 = as reproducible as the explainers themselves)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "fig_floor_cross_kappa.png"), dpi=160); plt.show()

## 15. Save outputs (no IDs, no raw rows)

In [ ]:
def attr_long(out):
    recs = []
    for (k, m, meth, ax_, r), arr in out["attr"].items():
        df = pd.DataFrame(arr, columns=out["vars"]); df.insert(0, "inst", range(len(arr)))
        df.insert(0, "run", r); df.insert(0, "axis", ax_); df.insert(0, "method", meth); df.insert(0, "model", m); df.insert(0, "fold", k)
        recs.append(df)
    return pd.concat(recs, ignore_index=True)

attr_long(MAIN).to_csv(os.path.join(OUT_DIR, "attributions_variable_level.csv"), index=False)
MAIN["inst"].to_csv(os.path.join(OUT_DIR, "explained_children.csv"), index=False)       # fold, local index, outcome, predicted probabilities only
MAIN["perf"].to_csv(os.path.join(OUT_DIR, "model_performance.csv"), index=False)
CROSS_TABLE.to_csv(os.path.join(OUT_DIR, "cross_method_vs_floor.csv"), index=False)
KAPPA_BY_FOLD.to_csv(os.path.join(OUT_DIR, "kappa_by_fold.csv"), index=False)
TIE_DIAG.to_csv(os.path.join(OUT_DIR, "tie_diagnostics.csv"), index=False)
FLOOR_TABLE.to_csv(os.path.join(OUT_DIR, "noise_floor.csv"), index=False)
IG_VARIANTS.to_csv(os.path.join(OUT_DIR, "ig_variants.csv"), index=False)
CROSS_MODEL.to_csv(os.path.join(OUT_DIR, "same_method_across_models.csv"), index=False)
NOISE_TABLE.to_csv(os.path.join(OUT_DIR, "noise_control.csv"), index=False)
GATE.to_csv(os.path.join(OUT_DIR, "gate_report.csv"), index=False)
MAIN["diag"].to_csv(os.path.join(OUT_DIR, "diagnostics.csv"), index=False)
from importlib import metadata as _md
def _ver(pkg):
    try: return _md.version(pkg)
    except Exception: return "n/a"
meta = dict(mode=MODE, feature_set=FEATURE_SET, vars=VARS, seeds=dict(split=SEED_SPLIT, inst=SEED_INST, noise=SEED_NOISE, boot=SEED_BOOT, null=SEED_NULL, bg=SEED_BG, ES=ES, TS=TS),
            R_E=R_E, R_T=R_T, n_per_fold=N_PER_FOLD, n_perm=N_PERM, b_shap=B_SHAP, lime_samples=LIME_SAMPLES, ig_steps=IG_STEPS, topk=TOPK, hp=HP, gates=GATES,
            n_analytic=int(len(y)), n_stunted=int(y.sum()), fold_sha256=FOLD_SHA, y_sha256=Y_SHA, weight_sum=float(w.sum()), ref_check=REF, projected_hours=proj_h, work_seconds=WORK, shap_min_effective_permutations=(min(SHAP_PERM_LOG) if SHAP_PERM_LOG else None), expected_fold_sha256=EXPECTED_FOLD_SHA256,
            versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__, torch=torch.__version__,
                          shap=shap.__version__, lime=_ver("lime"), captum=_ver("captum"), scipy=__import__("scipy").__version__))
meta["pin_command"] = "pip install " + " ".join(f"{p}=={_ver(p)}" for p in ["numpy", "pandas", "scikit-learn", "scipy", "torch", "shap", "lime", "captum"] if _ver(p) != "n/a")
json.dump(meta, open(os.path.join(OUT_DIR, "run_metadata.json"), "w"), indent=2, default=str)
print(sorted(os.listdir(OUT_DIR)))
try:
    import shutil; from google.colab import files
    shutil.make_archive("/content/protocol_pilot_outputs", "zip", OUT_DIR); files.download("/content/protocol_pilot_outputs.zip")
except Exception as e:
    print("zip/download skipped:", type(e).__name__)